# Lagrangian turbulence: data and MGD model
Data: one velocity component of the DNS of Li et al. (2024). Model: run `turbulence_ggd_mass_final`, seed 900
(wavelet marginals on two Morlet banks, Q = 1 and Q = 3; fourth-order scattering coefficients; $L^6$ norm).
$\Theta$ = $\Theta_{\rm reg}$ at the last time step of the transport ($t \approx 1$) with the constraint $L^6 < 0$, $\lambda = 5\cdot10^{-4}$.

Run on Jean Zay from `turbulence/`, after `python turbulence/talk_logp_generated.py` (GPU). Figures: `figures/talk/`.

In [ ]:
%matplotlib inline
import sys
from pathlib import Path
ROOT = Path.cwd()                                   # turbulence/
for p in ('codes', 'codes/filters', 'codes/potentials', 'data', 'turbulence', 'jets'):
    sys.path.insert(0, str(ROOT.parent / p))
import json
import numpy as np
import pandas as pd
import torch
from IPython.display import display
import talk_figures as tf
import run_comparison as rc
from logp_family_contrib import load_potentials
tf.set_output(ROOT / 'figures' / 'talk')          # every figure is also saved here as PNG

In [ ]:
LABEL, SEED = 'turbulence_ggd_mass_final', 900
runs, missing = rc.load_runs({'model': LABEL}, seed=SEED)
assert not missing, missing
run = runs['model']
cfg = run['cfg']
x1 = rc.reference_data(cfg).numpy()                  # training data (n, T), standardized
xt = run['xt'].numpy()                               # generated signals
d = np.load(ROOT / 'figures' / 'data' / f'logp_C_{LABEL}_lam0.0005.npz')                 # log p of the data
g = np.load(ROOT / 'figures' / 'data' / f'talk_logp_generated_{LABEL}.npz')              # log p of the generated
assert str(d['theta_desc']) == str(g['theta_desc']), (str(d['theta_desc']), str(g['theta_desc']))
assert str(d['theta_desc']).startswith('L_6<0'), d['theta_desc']
inc = np.abs(np.diff(x1, axis=1)).max(1)                                                 # same signals, same order
assert len(d['TOTAL']) == len(x1) and np.allclose(d['increment'], inc, rtol=1e-5), 'npz does not match x1'
theta = g['theta']
print('Theta:', d['theta_desc'])
print('data', x1.shape, '| generated', xt.shape)

## Data signals with rare events
A signal is rare when its largest one-step increment $|x(t+1)-x(t)|$ is above the 99th percentile over the data.
10 rare signals chosen at random.

In [ ]:
rare = d['increment'] > np.percentile(d['increment'], 99)
idx = np.random.default_rng(0).choice(np.flatnonzero(rare), 10, replace=False)
tf.plot_signals(x1[idx], 'data_rare_signals', color=tf.RARE)

## Wavelet coefficients of the data, per scale
Real part of the Morlet wavelet coefficients, one panel per scale $j$, divided by their standard deviation.
Dashed: standard Gaussian.

In [ ]:
tf.plot_wavelet_histograms(x1, int(cfg['J']), 'data_wavelet_histograms')

## Statistics of Li et al. (2024): data and model
Increments $\delta_\tau x = x(t+\tau) - x(t)$, structure functions $S_p(\tau) = \langle (\delta_\tau x)^p \rangle$,
flatness $F_p = S_p / S_2^{p/2}$, local exponents $\zeta_p = d\log S_p / d\log S_2$.
Bands: minimum to maximum over 10 disjoint batches of signals, as in Li et al.
$\tau_\eta$ = 9.33 samples of the DNS (Li et al., Table 1), i.e. 4.67 samples here (signals coarse-grained by 2).
Lags up to half the signal length (the signals are periodized windows).

In [ ]:
from paper_figures_li import TAU_ETA
tau_eta = TAU_ETA / (cfg['subseries_len'] // cfg['target_len'])
max_lag = x1.shape[-1] // 2
tf.plot_increment_pdfs(x1, xt, tau_eta, 'li_increment_pdfs')
tf.plot_structure_functions(x1, xt, tau_eta, max_lag, 'li_structure_functions')
tf.plot_flatness(x1, xt, tau_eta, max_lag, 'li_flatness')
tf.plot_local_exponents(x1, xt, tau_eta, max_lag, 'li_local_exponents')

## Distribution of log p over the data
$\log p(x) = \Theta \cdot \phi(x)$ up to a constant, shown relative to its median. Red: rare signals (as above).
Black line: Gumbel distribution of the minimum, fitted to all signals.

In [ ]:
tf.plot_logp_histogram(d['TOTAL'], 'logp_data', rare=rare, gumbel=True)

## Generated signals, from the rarest to the least rare
Left: $\log p$ of all generated signals, relative to the median. Arrows: 10 evenly spaced values from the lowest to
the highest $\log p$; right: the generated signal closest to each value, same colour, $\log p$ on top.

In [ ]:
tf.plot_ranked_signals(xt, g['logp'], 'generated_ranked', n=10)

## Number of statistics in the model
Per potential family, as used in the run (after removing duplicate filters and empty windows).

In [ ]:
from logp_family_contrib import phi_matrix
exp = rc._find_exp_dir(ROOT, run['config'])
pots = load_potentials(exp, cfg, 'cpu')
xs = torch.as_tensor(x1[:200]).reshape(200, 1, -1).float()
# check: log p of 200 data signals recomputed here = the saved values (same Theta, potentials and sign)
lp_check = phi_matrix(pots, xs) @ theta
assert np.allclose(lp_check, d['TOTAL'][:200], rtol=1e-6, atol=1e-6 * np.abs(lp_check).max()), 'saved log p differs'
print('log p of 200 data signals recomputed: matches the saved values')
n_stats = tf.count_statistics(pots, xs, theta, Q=cfg['Q'])
n_stats.to_csv(ROOT / 'figures' / 'talk' / 'number_of_statistics.csv')
display(n_stats)

## Contribution of each potential family to log p
For each data signal, $C_F(x) = \Theta_F \cdot \phi_F(x)$, and $\log p(x) = \sum_F C_F(x)$ + constant.
Top: mean over signals (its zero is arbitrary: a constant added to $\phi_F$ changes it, not $p$).
Middle: standard deviation across signals. Bottom: the same divided by the standard deviation of $\log p$.

In [ ]:
fams = [k for k in d.files if k not in ('increment', 'theta_desc', 'TOTAL')] + ['TOTAL']
tf.plot_energy_bars({k: d[k][None] for k in fams}, 'logp_per_family', Q=cfg['Q'])